In [6]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import os, time

if torch.cuda.is_available():
    device = "cuda"      # Colab GPU
elif torch.backends.mps.is_available():
    device = "mps"       # Mac GPU
else:
    device = "cpu"
print("Using:", device)

Using: mps


In [7]:
class SmallCNN(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, width, 3, padding=1), nn.BatchNorm2d(width), nn.ReLU(),
            nn.MaxPool2d(2),                                    # 32x32 -> 16x16
            nn.Conv2d(width, 2*width, 3, padding=1), nn.BatchNorm2d(2*width), nn.ReLU(),
            nn.MaxPool2d(2),                                    # 16x16 -> 8x8
            nn.Conv2d(2*width, 4*width, 3, padding=1), nn.BatchNorm2d(4*width), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),                            # 8x8 -> 1x1
        )
        self.classifier = nn.Linear(4*width, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

model = SmallCNN().to(device)
print(model)

SmallCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): AdaptiveAvgPool2d(output_size=1)
  )
  (classifier): Linear(in_features=128, out_features=10, bias=True)
)


In [8]:
for name, p in model.named_parameters():
    print(f"{name:25s} {str(list(p.shape)):20s} {p.numel():>8,}")
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

x = torch.randn(4, 3, 32, 32).to(device)
print("Output shape:", model(x).shape)

features.0.weight         [32, 3, 3, 3]             864
features.0.bias           [32]                       32
features.1.weight         [32]                       32
features.1.bias           [32]                       32
features.4.weight         [64, 32, 3, 3]         18,432
features.4.bias           [64]                       64
features.5.weight         [64]                       64
features.5.bias           [64]                       64
features.8.weight         [128, 64, 3, 3]        73,728
features.8.bias           [128]                     128
features.9.weight         [128]                     128
features.9.bias           [128]                     128
classifier.weight         [10, 128]               1,280
classifier.bias           [10]                       10

Total parameters: 94,986
Output shape: torch.Size([4, 10])


In [9]:
data_dir = "/content/data" if os.path.exists("/content") else os.path.expanduser("~/datasets") mean, std = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)

train_tf = T.Compose([T.RandomCrop(32, padding=4), T.RandomHorizontalFlip(),
                      T.ToTensor(), T.Normalize(mean, std)])
test_tf  = T.Compose([T.ToTensor(), T.Normalize(mean, std)])

train_set = torchvision.datasets.CIFAR10(data_dir, train=True,  download=True, transform=train_tf)
test_set  = torchvision.datasets.CIFAR10(data_dir, train=False, download=True, transform=test_tf)
train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256)

print("Train:", len(train_set), "| Test:", len(test_set))
print("Classes:", train_set.classes)

SyntaxError: invalid syntax (3149393509.py, line 1)

In [ ]:
def train_one_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total, correct, loss_sum = 0, 0, 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)            # 1. guess
        loss = loss_fn(outputs, labels)    # 2. how wrong?
        optimizer.zero_grad()
        loss.backward()                    # 3. find how to change each weight
        optimizer.step()                   # 4. change the weights a little
        loss_sum += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return loss_sum / total, 100 * correct / total

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        correct += (model(images).argmax(1) == labels).sum().item()
        total += images.size(0)
    return 100 * correct / total

In [ ]:
torch.manual_seed(0)
model = SmallCNN(width=32).to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)

for epoch in range(1, 6):
    t0 = time.time()
    tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, loss_fn)
    te_acc = evaluate(model, test_loader)
    print(f"Epoch {epoch} | loss {tr_loss:.3f} | train {tr_acc:.1f}% | test {te_acc:.1f}% | {time.time()-t0:.0f}s")

In [ ]:
torch.save(model.state_dict(), "smallcnn_w32.pth")
print("Saved model.")

if os.path.exists("/content"):          # on Colab → download it to your Mac
    from google.colab import files
    files.download("smallcnn_w32.pth")